<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/17-deep-learning-pytorch/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 17: Exercises

Twelve exercises: four Easy, five Medium and three Hard. The Easy ones are small tools you'll use all the time: counting parameters, accuracy from logits, a softmax that doesn't overflow, and a warmup-plus-cosine learning-rate schedule. The Medium ones build the pieces of a training script: gradients with autograd, a configurable MLP, cross-entropy from logits, and the training and evaluation loops. The Hard ones open up the black boxes, rewriting SGD with momentum, Adam (and AdamW) and batch normalisation so that they match PyTorch's own versions to many decimal places.

Most checks compare your code with PyTorch itself, and the others with a reference implementation of exactly the procedure described. Everything runs in a few seconds on a CPU; no GPU is needed.

**Run the setup cell first.**

Rules:

- PyTorch and NumPy. Don't call PyTorch's version of what you're writing (such as `F.cross_entropy` or `torch.optim.Adam`); the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_count_parameters(f):
    shared = nn.Linear(4, 4)
    frozen = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 2))
    for p in frozen[0].parameters():
        p.requires_grad = False
    mlp = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 10))
    _scenarios("count_parameters", [
        ("the example", lambda: f(nn.Linear(3, 2)), 8, "weights plus biases"),
        ("the lesson's 64-128-128-10 network", lambda: f(mlp), 26122, None),
        ("frozen layers are skipped", lambda: f(frozen), 12, "check p.requires_grad"),
        ("trainable_only=False counts everything", lambda: f(frozen, trainable_only=False), 67, None),
        ("a layer used twice is counted once", lambda: f(nn.Sequential(shared, nn.ReLU(), shared)), 20,
         "model.parameters() already skips duplicates"),
        ("batch norm buffers are not parameters", lambda: f(nn.BatchNorm1d(10)), 20, None),
        ("returns a Python int", lambda: type(f(nn.Linear(3, 2))).__name__, "int", None),
    ])


def check_accuracy_from_logits(f):
    g = torch.Generator().manual_seed(0)
    logits = torch.randn(40, 3, generator=g)
    targets = torch.randint(0, 3, (40,), generator=g)
    expected = float((logits.argmax(dim=1) == targets).double().mean())
    _scenarios("accuracy_from_logits", [
        ("the example", lambda: f(torch.tensor([[2.0, 1.0], [0.0, 3.0], [1.0, 0.5]]), torch.tensor([0, 1, 1])), 2 / 3,
         "the predicted class is the argmax of each row"),
        ("a random batch of 40", lambda: f(logits, targets), expected, None),
        ("negative logits", lambda: f(torch.tensor([[-5.0, -1.0, -3.0]]), torch.tensor([1])), 1.0, None),
        ("returns a Python float", lambda: type(f(logits, targets)).__name__, "float", "use .item()"),
        ("different lengths raise", lambda: f(logits, targets[:10]), Raises(ValueError), None),
    ])


def check_stable_softmax(f):
    g = torch.Generator().manual_seed(1)
    batch = torch.randn(5, 4, generator=g, dtype=torch.float64) * 3
    cube = torch.randn(2, 3, 4, generator=g, dtype=torch.float64)
    mixed = torch.tensor([[0.0, 1.0], [1000.0, 1001.0]], dtype=torch.float64)
    _scenarios("stable_softmax", [
        ("a single vector", lambda: f(torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)),
         torch.softmax(torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64), dim=0), None),
        ("a batch, one softmax per row", lambda: f(batch), torch.softmax(batch, dim=-1), "work along the last dimension"),
        ("large scores don't overflow", lambda: f(torch.tensor([1000.0, 1001.0])),
         torch.softmax(torch.tensor([1000.0, 1001.0]), dim=0), "subtract the largest value first"),
        ("very negative scores", lambda: f(torch.tensor([-1000.0, -1000.0])), torch.tensor([0.5, 0.5]), None),
        ("rows on very different scales", lambda: f(mixed), torch.softmax(mixed, dim=-1),
         "subtract the largest value of each row, not of the whole tensor"),
        ("three dimensions", lambda: f(cube), torch.softmax(cube, dim=-1), None),
    ])


def check_warmup_cosine_lr(f):
    weight = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.SGD([weight], lr=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=0.001)
    torch_rates = []
    for _ in range(20):
        torch_rates.append(optimizer.param_groups[0]["lr"])
        optimizer.step()
        scheduler.step()
    _scenarios("warmup_cosine_lr", [
        ("the first warmup step", lambda: f(0, 100, 0.1, warmup_steps=10), 0.01, "base_lr * (step + 1) / warmup_steps"),
        ("the end of warmup", lambda: f(9, 100, 0.1, warmup_steps=10), 0.1, None),
        ("the start of the decay", lambda: f(10, 100, 0.1, warmup_steps=10), 0.1, None),
        ("halfway through the decay", lambda: f(55, 100, 0.1, warmup_steps=10), 0.05, None),
        ("matches CosineAnnealingLR without warmup", lambda: [f(s, 20, 0.1, min_lr=0.001) for s in range(20)], torch_rates, None),
        ("stays at min_lr after the end", lambda: f(150, 100, 0.1, warmup_steps=10, min_lr=0.001), 0.001,
         "cap progress at 1"),
        ("returns a float", lambda: type(f(3, 20, 1, warmup_steps=5)).__name__, "float", None),
        ("warmup as long as training raises", lambda: f(0, 10, 0.1, warmup_steps=10), Raises(ValueError), None),
    ])


def check_gradient_at(f):
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    original = x.copy()
    A = np.array([[2.0, 1.0], [1.0, 3.0]])

    def unchanged():
        f(lambda t: (t ** 2).sum(), x)
        return x

    _scenarios("gradient_at", [
        ("the example", lambda: f(lambda t: (t ** 2).sum(), [1.0, 2.0, 3.0]), np.array([2.0, 4.0, 6.0]), None),
        ("a product", lambda: f(lambda t: t[0] * t[1] ** 3, [2.0, 1.5]), np.array([3.375, 13.5]), None),
        ("sin and exp", lambda: f(lambda t: torch.sin(t).sum() + torch.exp(t[0]), [0.5, 1.0]),
         np.array([math.cos(0.5) + math.exp(0.5), math.cos(1.0)]), None),
        ("a quadratic form", lambda: f(lambda t: t @ torch.tensor(A) @ t, [1.0, -1.0]), (A + A.T) @ np.array([1.0, -1.0]), None),
        ("keeps the shape of x", lambda: f(lambda t: (t ** 2).sum(), x), 2 * x, None),
        ("float64 precision", lambda: f(lambda t: ((t - 1e8) ** 2).sum(), [1e8 + 1]), np.array([2.0]),
         "in float32, 1e8 + 1 rounds to 1e8"),
        ("returns float64", lambda: str(f(lambda t: t.sum(), [1, 2]).dtype), "float64", None),
        ("x is not changed", unchanged, original, None),
        ("a vector output raises", lambda: f(lambda t: t * 2, [1.0, 2.0]), Raises(ValueError), None),
    ])


def check_make_mlp(f):
    def kinds(model):
        return [type(layer).__name__ for layer in model]

    def shape_out():
        torch.manual_seed(0)
        return tuple(f([64, 128, 128, 10])(torch.randn(5, 64)).shape)

    def sizes_of(model):
        return [(layer.in_features, layer.out_features) for layer in model if isinstance(layer, nn.Linear)]

    _scenarios("make_mlp", [
        ("returns an nn.Sequential", lambda: type(f([4, 8, 3])).__name__, "Sequential", None),
        ("one hidden layer", lambda: kinds(f([4, 8, 3])), ["Linear", "ReLU", "Linear"], "no ReLU after the last Linear"),
        ("layer sizes", lambda: sizes_of(f([64, 128, 128, 10])), [(64, 128), (128, 128), (128, 10)], None),
        ("number of parameters", lambda: sum(p.numel() for p in f([64, 128, 128, 10]).parameters()), 26122, None),
        ("output shape", shape_out, (5, 10), None),
        ("with dropout", lambda: kinds(f([4, 8, 8, 3], dropout=0.5)),
         ["Linear", "ReLU", "Dropout", "Linear", "ReLU", "Dropout", "Linear"], "a Dropout after every ReLU"),
        ("the dropout probability", lambda: [layer.p for layer in f([4, 8, 3], dropout=0.3) if isinstance(layer, nn.Dropout)], [0.3], None),
        ("no hidden layer", lambda: kinds(f([3, 2], dropout=0.5)), ["Linear"], None),
        ("a single size raises", lambda: f([5]), Raises(ValueError), None),
    ])


def check_cross_entropy_from_logits(f):
    g = torch.Generator().manual_seed(2)
    logits = torch.randn(8, 5, generator=g, dtype=torch.float64) * 2
    targets = torch.randint(0, 5, (8,), generator=g)
    big = torch.tensor([[1000.0, 0.0, -1000.0], [0.0, 2000.0, 10.0]], dtype=torch.float64)
    big_targets = torch.tensor([2, 1])

    def grad_of(loss_function):
        x = logits.clone().requires_grad_(True)
        loss_function(x, targets).backward()
        return x.grad

    _scenarios("cross_entropy_from_logits", [
        ("the example", lambda: f(torch.tensor([[2.0, 0.0]]), torch.tensor([0])), torch.tensor(math.log(1 + math.exp(-2.0))),
         "logsumexp(row) - row[true class]; return a tensor"),
        ("a random batch", lambda: f(logits, targets), F.cross_entropy(logits, targets), "the mean over examples"),
        ("returns a single value", lambda: tuple(f(logits, targets).shape), (), None),
        ("large logits stay finite", lambda: f(big, big_targets), F.cross_entropy(big, big_targets),
         "work with logs throughout; don't take the log of a softmax"),
        ("the gradient matches PyTorch's", lambda: grad_of(f), grad_of(F.cross_entropy), None),
    ])


def _toy_data(n=50, seed=3):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 4, generator=g)
    y = (X[:, 0] + X[:, 1] > 0).long() + (X[:, 2] > 1).long()
    return X, y


def _toy_model(dropout=0.0):
    torch.manual_seed(0)
    layers = [nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU()]
    if dropout:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(8, 3))
    return nn.Sequential(*layers)


def _reference_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, count = 0.0, 0
    for X, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
        count += len(y)
    return total / count


def _snapshot(model):
    return [t.detach().clone() for t in list(model.parameters()) + list(model.buffers()) if t.dtype.is_floating_point]


def check_train_one_epoch(f):
    X, y = _toy_data()

    def run(train_function, epochs=1):
        model = _toy_model()
        model.eval()
        loader = DataLoader(TensorDataset(X, y), batch_size=16)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
        losses = [train_function(model, loader, nn.CrossEntropyLoss(), optimizer) for _ in range(epochs)]
        return losses, _snapshot(model)

    expected_losses, expected_state = run(_reference_epoch, epochs=2)
    _scenarios("train_one_epoch", [
        ("the average loss of one epoch", lambda: run(f)[0][0], expected_losses[0],
         "weight each batch's loss by its number of examples"),
        ("returns a Python float", lambda: type(run(f)[0][0]).__name__, "float", "use loss.item()"),
        ("the weights and batch norm statistics after two epochs", lambda: run(f, epochs=2)[1], expected_state,
         "zero the gradients for every batch, and call model.train() first"),
        ("the loss of the second epoch", lambda: run(f, epochs=2)[0][1], expected_losses[1], None),
    ])


def check_evaluate_model(f):
    X, y = _toy_data(n=50, seed=4)
    trained = _toy_model(dropout=0.5)
    optimizer = torch.optim.Adam(trained.parameters(), lr=0.01)
    train_loader = DataLoader(TensorDataset(*_toy_data(n=200, seed=5)), batch_size=32)
    for _ in range(3):
        _reference_epoch(trained, train_loader, nn.CrossEntropyLoss(), optimizer)
    state = {k: v.clone() for k, v in trained.state_dict().items()}

    def fresh():
        trained.load_state_dict(state)
        trained.train()
        return trained

    loader = DataLoader(TensorDataset(X, y), batch_size=16)
    with torch.no_grad():
        model = fresh().eval()
        logits = model(X)
        expected = (F.cross_entropy(logits, y).item(), (logits.argmax(dim=1) == y).double().mean().item())

    def state_after():
        model = fresh()
        f(model, loader, nn.CrossEntropyLoss())
        return _snapshot(model)

    def graph_built():
        seen = []

        def recording_loss(logits, targets):
            seen.append(logits.requires_grad)
            return F.cross_entropy(logits, targets)

        f(fresh(), loader, recording_loss)
        return any(seen)

    def types():
        result = f(fresh(), loader, nn.CrossEntropyLoss())
        return [type(result).__name__] + [type(v).__name__ for v in result]

    _scenarios("evaluate_model", [
        ("loss and accuracy", lambda: f(fresh(), loader, nn.CrossEntropyLoss()), expected,
         "call model.eval(), and weight each batch by its size"),
        ("returns a tuple of two floats", types, ["tuple", "float", "float"], None),
        ("weights and batch norm statistics unchanged", state_after, _snapshot(fresh()), "evaluation mode stops the running statistics from updating"),
        ("no gradients are tracked", graph_built, False, "use torch.no_grad()"),
    ])


def check_sgd_momentum_step(f):
    g = torch.Generator().manual_seed(6)
    shapes = [(3, 2), (4,)]
    start = [torch.randn(s, generator=g, dtype=torch.float64) for s in shapes]
    gradients = [[torch.randn(s, generator=g, dtype=torch.float64) for s in shapes] for _ in range(5)]

    def with_torch(steps, momentum, weight_decay, lr=0.05):
        params = [p.clone().requires_grad_(True) for p in start]
        optimizer = torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
        for grads in gradients[:steps]:
            for p, grad in zip(params, grads):
                p.grad = grad.clone()
            optimizer.step()
        buffers = [optimizer.state[p].get("momentum_buffer") for p in params]
        return [p.detach() for p in params], buffers

    def with_yours(steps, momentum, weight_decay, lr=0.05):
        params = [p.clone() for p in start]
        velocities = [torch.zeros_like(p) for p in start]
        for grads in gradients[:steps]:
            f(params, [grad.clone() for grad in grads], velocities, lr, momentum=momentum, weight_decay=weight_decay)
        return params, velocities

    _scenarios("sgd_momentum_step", [
        ("one step from rest is plain SGD", lambda: with_yours(1, 0.9, 0.0)[0], [p - 0.05 * grad for p, grad in zip(start, gradients[0])],
         "update the tensors in place, for example with p.sub_(...)"),
        ("five steps match torch.optim.SGD", lambda: with_yours(5, 0.9, 0.0)[0], with_torch(5, 0.9, 0.0)[0], None),
        ("the velocities match PyTorch's momentum buffers", lambda: with_yours(5, 0.9, 0.0)[1], with_torch(5, 0.9, 0.0)[1],
         "v = momentum * v + d, with the learning rate applied afterwards"),
        ("with weight decay", lambda: with_yours(5, 0.9, 0.01)[0], with_torch(5, 0.9, 0.01)[0], None),
        ("a different momentum", lambda: with_yours(4, 0.5, 0.0, lr=0.2)[0], with_torch(4, 0.5, 0.0, lr=0.2)[0], None),
    ])


def check_adam_step(f):
    g = torch.Generator().manual_seed(7)
    start = torch.randn(3, 4, generator=g, dtype=torch.float64)
    gradients = [torch.randn(3, 4, generator=g, dtype=torch.float64) for _ in range(10)]
    tiny = [grad * 1e-9 for grad in gradients]

    def with_torch(steps, grads=gradients, lr=0.01, weight_decay=0.0, decoupled=False):
        param = start.clone().requires_grad_(True)
        cls = torch.optim.AdamW if decoupled else torch.optim.Adam
        optimizer = cls([param], lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay)
        for grad in grads[:steps]:
            param.grad = grad.clone()
            optimizer.step()
        state = optimizer.state[param]
        return param.detach(), state["exp_avg"], state["exp_avg_sq"]

    def with_yours(steps, grads=gradients, lr=0.01, weight_decay=0.0, decoupled=False):
        param, m, v = start.clone(), torch.zeros_like(start), torch.zeros_like(start)
        for t, grad in enumerate(grads[:steps], start=1):
            f(param, grad.clone(), m, v, t, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, decoupled=decoupled)
        return param, m, v

    _scenarios("adam_step", [
        ("the first step moves each weight by about lr", lambda: with_yours(1)[0], with_torch(1)[0],
         "bias correction makes the first step lr * g / (|g| + eps)"),
        ("ten steps match torch.optim.Adam", lambda: with_yours(10)[0], with_torch(10)[0], None),
        ("m and v match PyTorch's running averages", lambda: with_yours(10)[1:], with_torch(10)[1:], None),
        ("tiny gradients, where eps matters", lambda: with_yours(5, grads=tiny)[0], with_torch(5, grads=tiny)[0],
         "add eps after the square root"),
        ("Adam's weight decay", lambda: with_yours(10, weight_decay=0.1)[0], with_torch(10, weight_decay=0.1)[0], None),
        ("AdamW's decoupled weight decay", lambda: with_yours(10, weight_decay=0.1, decoupled=True)[0],
         with_torch(10, weight_decay=0.1, decoupled=True)[0], "shrink the weights directly, before the Adam step"),
        ("t = 0 raises", lambda: f(start.clone(), gradients[0], torch.zeros_like(start), torch.zeros_like(start), 0), Raises(ValueError), None),
    ])


def check_batchnorm1d_forward(f):
    g = torch.Generator().manual_seed(8)
    batches = [torch.randn(6, 3, generator=g, dtype=torch.float64) * 2 + 1 for _ in range(3)]
    gamma0 = torch.tensor([1.5, 0.5, -1.0], dtype=torch.float64)
    beta0 = torch.tensor([0.1, -0.2, 0.3], dtype=torch.float64)

    def torch_layer(momentum=0.1):
        layer = nn.BatchNorm1d(3, momentum=momentum).double()
        with torch.no_grad():
            layer.weight.copy_(gamma0)
            layer.bias.copy_(beta0)
        return layer

    def yours_train(momentum=0.1):
        running_mean, running_var = torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64)
        outputs = [f(x, gamma0, beta0, running_mean, running_var, True, momentum=momentum) for x in batches]
        return outputs, running_mean, running_var

    def torch_train(momentum=0.1):
        layer = torch_layer(momentum)
        outputs = [layer(x).detach() for x in batches]
        return outputs, layer.running_mean, layer.running_var

    def yours_eval():
        _, running_mean, running_var = yours_train()
        return f(batches[0][:2] * 3, gamma0, beta0, running_mean, running_var, False)

    def torch_eval():
        layer = torch_layer()
        for x in batches:
            layer(x)
        layer.eval()
        return layer(batches[0][:2] * 3).detach()

    def eval_stats():
        _, running_mean, running_var = yours_train()
        before = [running_mean.clone(), running_var.clone()]
        f(batches[0], gamma0, beta0, running_mean, running_var, False)
        return [running_mean, running_var], before

    def gradients(fn):
        x = batches[0].clone().requires_grad_(True)
        gamma = gamma0.clone().requires_grad_(True)
        beta = beta0.clone().requires_grad_(True)
        weights = torch.arange(18, dtype=torch.float64).reshape(6, 3)
        (fn(x, gamma, beta) * weights).sum().backward()
        return [x.grad, gamma.grad, beta.grad]

    def yours_fn(x, gamma, beta):
        return f(x, gamma, beta, torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64), True)

    def torch_fn(x, gamma, beta):
        return F.batch_norm(x, torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64), gamma, beta, training=True)

    _scenarios("batchnorm1d_forward", [
        ("training-mode output", lambda: yours_train()[0][0], torch_train()[0][0], "normalise with the biased variance of the batch"),
        ("running statistics after three batches", lambda: list(yours_train()[1:]), list(torch_train()[1:]),
         "the running variance uses the unbiased variance, var * n / (n - 1)"),
        ("a different momentum", lambda: list(yours_train(0.3)[1:]), list(torch_train(0.3)[1:]), None),
        ("evaluation mode uses the running statistics", yours_eval, torch_eval(), None),
        ("evaluation mode leaves the statistics alone", lambda: eval_stats()[0], eval_stats()[1], None),
        ("gradients flow through the batch statistics", lambda: gradients(yours_fn), gradients(torch_fn),
         "don't detach the batch mean or variance"),
        ("a batch of one raises", lambda: f(batches[0][:1], gamma0, beta0, torch.zeros(3, dtype=torch.float64),
                                            torch.ones(3, dtype=torch.float64), True), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `count_parameters`

Write `count_parameters(model, trainable_only=True)` that returns the number of parameters (individual numbers) in a PyTorch model, as a Python `int`.

- With `trainable_only=True`, count only parameters with `requires_grad=True`. Frozen parameters are skipped.
- With `trainable_only=False`, count every parameter.

Example: `nn.Linear(3, 2)` has a 2 x 3 weight matrix and 2 biases, so `count_parameters(nn.Linear(3, 2))` is `8`.

In [2]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    # your code here
    pass

In [3]:
check_count_parameters(count_parameters)

FAIL  count_parameters: the example
      expected: 8
      got:      None
      hint: weights plus biases


### Exercise 2: `accuracy_from_logits`

Write `accuracy_from_logits(logits, targets)` that returns the fraction of correct predictions as a Python `float`.

- `logits` has shape `(n, n_classes)`: one row of raw scores per example. The predicted class is the column with the largest score.
- `targets` has shape `(n,)` and holds the true class numbers.
- Raise a `ValueError` if `logits` and `targets` have different lengths.

Example: `accuracy_from_logits(torch.tensor([[2.0, 1.0], [0.0, 3.0], [1.0, 0.5]]), torch.tensor([0, 1, 1]))` is `2 / 3`.

In [4]:
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    # your code here
    pass

In [5]:
check_accuracy_from_logits(accuracy_from_logits)

FAIL  accuracy_from_logits: the example
      expected: 0.6666666666666666
      got:      None
      hint: the predicted class is the argmax of each row


### Exercise 3: `stable_softmax`

Write `stable_softmax(logits)` that applies softmax along the **last** dimension of a tensor, without overflowing for large scores.

- Each value becomes `exp(z) / sum(exp(z))`, where the sum runs over the last dimension.
- `exp(1000)` is infinity in floating point, so first subtract the **largest value in each row** (each slice along the last dimension). This doesn't change the answer, because the factor `exp(-max)` cancels.
- The result has the same shape as the input.
- Don't call `torch.softmax` or `F.softmax`; the check uses them to test your answer.

Example: `stable_softmax(torch.tensor([1000.0, 1001.0]))` is about `[0.2689, 0.7311]`, not `[nan, nan]`.

In [6]:
def stable_softmax(logits: torch.Tensor) -> torch.Tensor:
    # your code here
    pass

In [7]:
check_stable_softmax(stable_softmax)

FAIL  stable_softmax: a single vector
      expected: tensor(shape=(3,), dtype=float64, [0.09 , 0.2447, 0.6652])
      got:      None


### Exercise 4: `warmup_cosine_lr`

Write `warmup_cosine_lr(step, total_steps, base_lr, warmup_steps=0, min_lr=0.0)` that returns the learning rate for a given step (counting from 0), as a `float`. This is the schedule used to train most large networks.

- **Warmup**: while `step < warmup_steps`, the rate rises linearly: `base_lr * (step + 1) / warmup_steps`.
- **Cosine decay**: after that, let `progress = (step - warmup_steps) / (total_steps - warmup_steps)`, capped at 1. The rate is `min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(pi * progress))`.
- Raise a `ValueError` if `warmup_steps >= total_steps`.

With `warmup_steps=0` this matches PyTorch's `CosineAnnealingLR`. Example: `warmup_cosine_lr(0, 100, 0.1, warmup_steps=10)` is `0.01`, and `warmup_cosine_lr(10, 100, 0.1, warmup_steps=10)` is `0.1`.

In [8]:
def warmup_cosine_lr(step: int, total_steps: int, base_lr: float, warmup_steps: int = 0, min_lr: float = 0.0) -> float:
    # your code here
    pass

In [9]:
check_warmup_cosine_lr(warmup_cosine_lr)

FAIL  warmup_cosine_lr: the first warmup step
      expected: 0.01
      got:      None
      hint: base_lr * (step + 1) / warmup_steps


## Medium

### Exercise 5: `gradient_at`

Write `gradient_at(f, x)` that uses autograd to compute the gradient of a function at a point.

- `f` takes a tensor and returns a single number (a tensor with one element) built from PyTorch operations.
- `x` is a list or NumPy array of any shape.
- Return the gradient as a NumPy array of `float64`, with the same shape as `x`.
- Work in `float64` throughout, so the answer is accurate even for large numbers.
- Raise a `ValueError` if `f` returns more than one number.
- Don't change `x`.

Example: `gradient_at(lambda t: (t ** 2).sum(), [1.0, 2.0, 3.0])` is `array([2., 4., 6.])`.

In [10]:
def gradient_at(f, x) -> np.ndarray:
    # your code here
    pass

In [11]:
check_gradient_at(gradient_at)

FAIL  gradient_at: the example
      expected: array(shape=(3,), dtype=float64, [2., 4., 6.])
      got:      None


### Exercise 6: `make_mlp`

Write `make_mlp(sizes, dropout=0.0)` that builds a multilayer perceptron as an `nn.Sequential`.

- `sizes` lists the layer widths, from the number of inputs to the number of outputs. For example `[64, 128, 10]` means 64 inputs, one hidden layer of 128 units and 10 outputs.
- Use one `nn.Linear` for each consecutive pair of sizes.
- Put an `nn.ReLU()` after every `nn.Linear` **except the last** (the output should be raw logits).
- If `dropout > 0`, put an `nn.Dropout(dropout)` after every ReLU.
- Raise a `ValueError` if `sizes` has fewer than 2 entries.

Example: `make_mlp([4, 8, 3], dropout=0.5)` is `Sequential(Linear(4, 8), ReLU(), Dropout(0.5), Linear(8, 3))`.

In [12]:
def make_mlp(sizes, dropout: float = 0.0) -> nn.Sequential:
    # your code here
    pass

In [13]:
check_make_mlp(make_mlp)

FAIL  make_mlp: returns an nn.Sequential
      expected: 'Sequential'
      got:      'NoneType'


### Exercise 7: `cross_entropy_from_logits`

Write `cross_entropy_from_logits(logits, targets)` that computes the mean cross-entropy loss, exactly like `F.cross_entropy` with its default settings.

- `logits` has shape `(n, n_classes)`; `targets` holds the true class of each example.
- The loss for one example is `-log(softmax(row)[true class])`, which simplifies to `logsumexp(row) - row[true class]`.
- Return the **mean** over the examples, as a tensor with a single value (shape `()`), so that `backward()` works on it.
- It must not overflow for large logits. You may use `torch.logsumexp`, which is stable.
- Don't call `F.cross_entropy`, `nn.CrossEntropyLoss`, `F.log_softmax` or `F.nll_loss`.

Example: for `logits = torch.tensor([[2.0, 0.0]])` and `targets = torch.tensor([0])`, the loss is `log(1 + e**-2)`, about `0.1269`.

In [14]:
def cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # your code here
    pass

In [15]:
check_cross_entropy_from_logits(cross_entropy_from_logits)

FAIL  cross_entropy_from_logits: the example
      expected: tensor(shape=(), dtype=float32, 0.1269)
      got:      None
      hint: logsumexp(row) - row[true class]; return a tensor


### Exercise 8: `train_one_epoch`

Write `train_one_epoch(model, loader, loss_fn, optimizer)` that trains a model for one pass over a `DataLoader` and returns the average training loss as a `float`.

- Put the model in **training mode** first (this matters for dropout and batch normalisation).
- For every batch `(X, y)`: clear the old gradients, compute the loss, backpropagate, and take an optimiser step.
- Return the average loss **per example**. The last batch can be smaller than the others, so weight each batch's loss by its number of examples.

Your loop should do exactly what the lesson's `train_epoch` does; the check compares the trained weights with a reference loop.

In [16]:
def train_one_epoch(model: nn.Module, loader, loss_fn, optimizer) -> float:
    # your code here
    pass

In [17]:
check_train_one_epoch(train_one_epoch)

FAIL  train_one_epoch: the average loss of one epoch
      expected: 1.081517720222473
      got:      None
      hint: weight each batch's loss by its number of examples


### Exercise 9: `evaluate_model`

Write `evaluate_model(model, loader, loss_fn)` that returns `(mean_loss, accuracy)` for a model on a `DataLoader`, both as `float`.

- Put the model in **evaluation mode**, so dropout is off and batch normalisation uses its running statistics.
- Don't track gradients: wrap the work in `torch.no_grad()` (or decorate the function with `@torch.no_grad()`).
- Weight each batch by its number of examples, as in `train_one_epoch`.
- The predicted class is the largest logit.
- Evaluating must not change the model's weights or its batch normalisation statistics.

In [18]:
def evaluate_model(model: nn.Module, loader, loss_fn):
    # your code here
    pass

In [19]:
check_evaluate_model(evaluate_model)

FAIL  evaluate_model: loss and accuracy
      expected: (0.8058632612228394, 0.68)
      got:      None
      hint: call model.eval(), and weight each batch by its size


## Hard

### Exercise 10: `sgd_momentum_step`

Write `sgd_momentum_step(params, grads, velocities, lr, momentum=0.9, weight_decay=0.0)` that performs one step of SGD with momentum, giving exactly the same results as `torch.optim.SGD`.

`params`, `grads` and `velocities` are lists of tensors with matching shapes. For each parameter `p` with gradient `g` and velocity `v`:

1. `d = g + weight_decay * p`
2. `v = momentum * v + d`
3. `p = p - lr * v`

Update `params` and `velocities` **in place** (with operations such as `mul_`, `add_` and `sub_`), so the caller's tensors change. Return nothing. Velocities start at zero.

In [20]:
def sgd_momentum_step(params, grads, velocities, lr: float, momentum: float = 0.9, weight_decay: float = 0.0) -> None:
    # your code here
    pass

In [21]:
check_sgd_momentum_step(sgd_momentum_step)

FAIL  sgd_momentum_step: one step from rest is plain SGD
      expected: [tensor(shape=(3, 2), dtype=float64, [[-1.8661, -1.0133], [ 0.6637, -0.6762], [-1.418 , ...,
           tensor(shape=(4,), dtype=float64, [ 1.4225, 0.6248, 1.9499, -1.1302])]
      got:      [tensor(shape=(3, 2), dtype=float64, [[-1.8744, -0.9937], [ 0.7185, -0.6985], [-1.4716, ...,
           tensor(shape=(4,), dtype=float64, [ 1.4557, 0.6403, 1.9355, -1.1402])]
      hint: update the tensors in place, for example with p.sub_(...)


### Exercise 11: `adam_step`

Write `adam_step(param, grad, m, v, t, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False)` that performs one Adam update on a single tensor, matching `torch.optim.Adam` (or `torch.optim.AdamW` when `decoupled=True`).

`m` and `v` start as zeros, and `t` is the step number, starting at 1. With `beta1, beta2 = betas`:

1. **Weight decay.** If `decoupled` is False (Adam), add it to the gradient: `g = grad + weight_decay * param`. If `decoupled` is True (AdamW), shrink the weights directly instead: `param = param * (1 - lr * weight_decay)`, and use `g = grad`.
2. `m = beta1 * m + (1 - beta1) * g`
3. `v = beta2 * v + (1 - beta2) * g * g`
4. Bias correction: `m_hat = m / (1 - beta1 ** t)` and `v_hat = v / (1 - beta2 ** t)`
5. `param = param - lr * m_hat / (sqrt(v_hat) + eps)`

Update `param`, `m` and `v` **in place**, and return nothing. Raise a `ValueError` if `t < 1`.

In [22]:
def adam_step(param, grad, m, v, t: int, lr: float = 1e-3, betas=(0.9, 0.999), eps: float = 1e-8,
              weight_decay: float = 0.0, decoupled: bool = False) -> None:
    # your code here
    pass

In [23]:
check_adam_step(adam_step)

FAIL  adam_step: the first step moves each weight by about lr
      expected: tensor(shape=(3, 4), dtype=float64, [[-0.1568, 0.7761, 0.9368, -1.1243], [ 1.6808, -0.9...
      got:      tensor(shape=(3, 4), dtype=float64, [[-0.1468, 0.7861, 0.9468, -1.1143], [ 1.6908, -0.8...
      hint: bias correction makes the first step lr * g / (|g| + eps)


### Exercise 12: `batchnorm1d_forward`

Write `batchnorm1d_forward(x, gamma, beta, running_mean, running_var, training, momentum=0.1, eps=1e-5)` that does what `nn.BatchNorm1d` does in its forward pass, for an input of shape `(batch, features)`.

**Training mode** (`training=True`):

- Compute each feature's mean and variance over the batch. Use the **biased** variance (divide by `n`) to normalise.
- Return `gamma * (x - mean) / sqrt(var + eps) + beta`.
- Update the running statistics in place: `running_mean = (1 - momentum) * running_mean + momentum * mean`, and the same for `running_var`, but using the **unbiased** variance (divide by `n - 1`).
- Raise a `ValueError` if the batch has fewer than 2 examples.

**Evaluation mode** (`training=False`): normalise with `running_mean` and `running_var` instead, and don't change them.

Gradients must flow back to `x`, `gamma` and `beta` through the batch statistics, just as in PyTorch.

In [24]:
def batchnorm1d_forward(x, gamma, beta, running_mean, running_var, training: bool,
                        momentum: float = 0.1, eps: float = 1e-5) -> torch.Tensor:
    # your code here
    pass

In [25]:
check_batchnorm1d_forward(batchnorm1d_forward)

FAIL  batchnorm1d_forward: training-mode output
      expected: tensor(shape=(6, 3), dtype=float64, [[-2.6131, -0.0712, -0.5962], [-0.5385, -0.7605, -0...
      got:      None
      hint: normalise with the biased variance of the batch


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, which trains a network to recognise photos of clothing and takes it from a linear baseline to a regularised MLP.